<a href="https://colab.research.google.com/github/mtalhak003/AI-ML-Deep-Learning/blob/main/Parthenium_Model_Confusion_matrix.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =====================================================================
# Parthenium RF-DETR Small — Confusion Matrix Generation (Colab)
# Updated version: verifies class list, forces clean re-download,
# and does NOT assume a fixed matrix layout (avoids silent index bugs).
# =====================================================================
#
# BEFORE RUNNING:
#   1. Runtime -> Restart runtime  (clears any cached/stale state)
#   2. Regenerate a NEW dataset version in Roboflow (Versions tab ->
#      Generate New Version) with the stray "weed-pGwV" class removed,
#      and confirm at the class-selection step that only "Parthenium"
#      is listed. Note the new version number and put it below.
#   3. Paste in a FRESH Roboflow API key. Do not reuse any key that
#      has ever been pasted into a shared notebook/chat before —
#      regenerate it from Roboflow account settings first.

!pip install roboflow supervision -q

from roboflow import Roboflow
import supervision as sv
import numpy as np
import os
import shutil

# ---- CONFIG ----
ROBOFLOW_API_KEY = "PASTCvXTVkDmEzc3sznplEwqE"
WORKSPACE = "aiagri-3ayg4"
PROJECT = "parthenium-j9fie"
DATASET_VERSION = None   # <-- set this to the NEW version number you generated
MODEL_VERSION = 3        # the trained model version you want to evaluate
CONFIDENCE_THRESHOLD = 40  # % — match whatever threshold you report as P/R
OVERLAP_THRESHOLD = 30     # % — NMS overlap for the hosted inference API
IOU_MATCH_THRESHOLD = 0.5  # matches your mAP@50 reporting

assert DATASET_VERSION is not None, (
    "Set DATASET_VERSION to your newly generated dataset version number "
    "before running (see step 2 above)."
)

# ---- 0. Force a clean slate: remove any old downloaded dataset folders ----
# Roboflow typically names the download folder "<project-name>-<version>"
for name in os.listdir("."):
    if PROJECT in name or name.lower().startswith("parthenium"):
        print(f"Removing stale local folder: {name}")
        shutil.rmtree(name, ignore_errors=True)

# ---- 1. Download the validation dataset fresh (COCO format, with ground truth) ----
rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace(WORKSPACE).project(PROJECT)
dataset = project.version(DATASET_VERSION).download("coco")

# ---- 2. Load ground-truth annotations for the validation split ----
val_dir = os.path.join(dataset.location, "valid")
coco_json_path = os.path.join(val_dir, "_annotations.coco.json")

ground_truth = sv.DetectionDataset.from_coco(
    images_directory_path=val_dir,
    annotations_path=coco_json_path,
)

print(f"Loaded {len(ground_truth)} validation images with ground truth.")

# ---- 2b. VERIFY the class list before doing anything else ----
print("Ground-truth classes found in this dataset version:", ground_truth.classes)
if list(ground_truth.classes) != ["Parthenium"]:
    raise RuntimeError(
        "Ground-truth class list is not exactly ['Parthenium']. "
        "The stale 'weed-pGwV' class is likely still baked into this "
        "dataset version's annotation file. Go back to Roboflow, "
        "confirm the class is removed at Project Settings AND at "
        "version-generation time, generate a new version, and update "
        "DATASET_VERSION above before re-running."
    )
print("Class list verified as clean. Proceeding.")

# ---- 3. Run your trained model on each validation image ----
model = project.version(MODEL_VERSION).model

# Also verify the deployed model's own predicted class name matches,
# using the first validation image, before running the full loop.
first_image_name = list(ground_truth.images.keys())[0] if hasattr(ground_truth, "images") else None

predictions_list = []
targets_list = []
seen_pred_classes = set()

for image_name, image, gt_detections in ground_truth:
    image_path = os.path.join(val_dir, image_name)

    result = model.predict(
        image_path,
        confidence=CONFIDENCE_THRESHOLD,
        overlap=OVERLAP_THRESHOLD,
    ).json()

    for pred in result.get("predictions", []):
        seen_pred_classes.add(pred["class"])

    # Convert Roboflow API response -> supervision Detections
    pred_detections = sv.Detections.from_inference(result)

    predictions_list.append(pred_detections)
    targets_list.append(gt_detections)

print("Class names seen in model predictions across the whole validation set:", seen_pred_classes)
if seen_pred_classes and seen_pred_classes != {"Parthenium"}:
    print(
        "WARNING: the model is predicting class names other than "
        "'Parthenium' (e.g. a stale label like 'weed-pGwV'). This will "
        "cause correctly-localized boxes to be miscounted as class "
        "mismatches in the confusion matrix rather than true positives. "
        "Consider retraining a fresh model version, or remap labels "
        "before building the matrix (see commented block below)."
    )

# Optional: uncomment to force-remap any stale predicted label before
# building the matrix, IF you've confirmed via the check above that the
# only issue is a leftover label string (not an actual detection problem).
#
# for preds in predictions_list:
#     preds.data if hasattr(preds, "data") else None  # no-op placeholder
# (supervision's Detections class name field lives outside `.data` in
#  most versions -- if remapping is needed, do it on the raw `result`
#  dict's `pred["class"]` BEFORE calling sv.Detections.from_inference,
#  inside the loop above.)

# ---- 4. Build the confusion matrix ----
class_names = list(ground_truth.classes)  # should be ["Parthenium"]

confusion_matrix = sv.ConfusionMatrix.from_detections(
    predictions=predictions_list,
    targets=targets_list,
    classes=class_names,
    conf_threshold=CONFIDENCE_THRESHOLD / 100,
    iou_threshold=IOU_MATCH_THRESHOLD,
)

print("Confusion matrix class order:", confusion_matrix.classes)
print("Raw matrix (rows = predicted, cols = ground truth, including background row/col):")
print(confusion_matrix.matrix)

# ---- 5. Plot and save it ----
confusion_matrix.plot()

# ---- 6. Derive precision / recall / F1 WITHOUT assuming a fixed shape ----
# `supervision` appends a final "background" row/column automatically, so
# for N real classes the matrix is (N+1) x (N+1). For our single-class
# case (N=1) that means a 2x2 matrix:
#   matrix[0][0] = TP  (Parthenium predicted, Parthenium true)
#   matrix[0][1] = FP  (Parthenium predicted, background true)
#   matrix[1][0] = FN  (background predicted, Parthenium true)
# We assert the shape instead of assuming it, so a leftover stray class
# would raise an error here rather than silently producing wrong numbers.
matrix = np.array(confusion_matrix.matrix)
expected_shape = (len(class_names) + 1, len(class_names) + 1)
assert matrix.shape == expected_shape, (
    f"Unexpected confusion matrix shape {matrix.shape}, expected "
    f"{expected_shape} for {len(class_names)} class(es) + background. "
    "This usually means an extra stray class is still present -- "
    "check confusion_matrix.classes above."
)

tp = matrix[0][0]
fp = matrix[0][1]
fn = matrix[1][0]

precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

print(f"TP={tp}, FP={fp}, FN={fn}")
print(f"Precision={precision:.3f}, Recall={recall:.3f}, F1={f1:.3f}")
print(
    "Compare these against the 94.7% / 87.8% / 91.1% reported on the "
    "Roboflow dashboard -- some gap is expected since this script uses "
    "a fixed confidence threshold, while the dashboard reports the "
    "best-threshold value from the full precision-recall curve."
)

AssertionError: Set DATASET_VERSION to your newly generated dataset version number before running (see step 2 above).

In [ ]:
result = model.predict(image_path, confidence=CONFIDENCE_THRESHOLD, overlap=OVERLAP_THRESHOLD).json()
print(result["predictions"][0]["class"])

Parthenium
